# 20. Instacart 데이터 로딩 및 최적화

## 개요

Kaggle Instacart Market Basket Analysis 데이터셋을 로드하고 최적화된 형태로 저장.

**데이터셋 정보**:
- 3,421,083 주문 (orders)
- 206,209 사용자 (users)
- 49,688 상품 (products)
- 32,434,489 주문-상품 관계 (order_products)

**학술적 근거**:
- Kaggle Competition: Instacart Market Basket Analysis (2017)
- 식료품 이커머스 추천 시스템의 표준 벤치마크 데이터셋

**출력 파일**:
- `data/instacart/orders.parquet`
- `data/instacart/order_products.parquet`
- `data/instacart/products.parquet`
- `data/instacart/aisles.parquet`
- `data/instacart/departments.parquet`

## 1. 환경 설정

In [1]:
# 필수 라이브러리 임포트
import os
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

# 프로젝트 루트 경로 설정
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))

# 유틸리티 임포트
from utils.personalization import InstacartDataLoader

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"현재 시간: {datetime.now().isoformat()}")

프로젝트 루트: d:\VSC_Project\SSAFY\SSAFY_Class_18_Team_4_Final_Capstone\SSAFY_Class_18_Team_4_Final_Capstone
현재 시간: 2025-12-22T00:04:01.547529


In [2]:
# 경로 설정
# Kaggle에서 다운로드한 Instacart 데이터 경로
# https://www.kaggle.com/c/instacart-market-basket-analysis/data

INSTACART_RAW_PATH = PROJECT_ROOT / 'data' / 'instacart_raw'
INSTACART_OUTPUT_PATH = PROJECT_ROOT / 'data' / 'instacart'

# 출력 디렉토리 생성
INSTACART_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f"원본 데이터 경로: {INSTACART_RAW_PATH}")
print(f"출력 경로: {INSTACART_OUTPUT_PATH}")

# 원본 파일 확인
expected_files = [
    'orders.csv',
    'order_products__prior.csv',
    'order_products__train.csv',
    'products.csv',
    'aisles.csv',
    'departments.csv',
]

print("\n--- 원본 파일 확인 ---")
for file in expected_files:
    file_path = INSTACART_RAW_PATH / file
    exists = file_path.exists()
    size = file_path.stat().st_size / (1024**2) if exists else 0
    print(f"  {file}: {'✓' if exists else '✗'} ({size:.1f} MB)")

원본 데이터 경로: d:\VSC_Project\SSAFY\SSAFY_Class_18_Team_4_Final_Capstone\SSAFY_Class_18_Team_4_Final_Capstone\data\instacart_raw
출력 경로: d:\VSC_Project\SSAFY\SSAFY_Class_18_Team_4_Final_Capstone\SSAFY_Class_18_Team_4_Final_Capstone\data\instacart

--- 원본 파일 확인 ---
  orders.csv: ✓ (103.9 MB)
  order_products__prior.csv: ✓ (550.8 MB)
  order_products__train.csv: ✓ (23.5 MB)
  products.csv: ✓ (2.1 MB)
  aisles.csv: ✓ (0.0 MB)
  departments.csv: ✓ (0.0 MB)


## 2. 데이터 로더 초기화

In [3]:
# Instacart 데이터 로더 초기화
loader = InstacartDataLoader(
    data_path=str(INSTACART_RAW_PATH),
    chunk_size=100000,  # 메모리 효율적인 청크 처리
)

print("데이터 로더 초기화 완료")
print(f"청크 크기: {loader.chunk_size:,}")

데이터 로더 초기화 완료
청크 크기: 100,000


## 3. 주문 데이터 로드 (orders.csv)

In [4]:
%%time

# 주문 데이터 로드
orders_df = loader.load_orders()

print(f"\n주문 데이터 로드 완료")
print(f"  - 총 주문 수: {len(orders_df):,}")
print(f"  - 고유 사용자 수: {orders_df['user_id'].nunique():,}")
print(f"  - 메모리 사용량: {orders_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

INFO:utils.personalization.data_processor:orders.csv 로드 중... (d:\VSC_Project\SSAFY\SSAFY_Class_18_Team_4_Final_Capstone\SSAFY_Class_18_Team_4_Final_Capstone\data\instacart_raw\orders.csv)
INFO:utils.personalization.data_processor:  로드 완료: 3,421,083개 레코드 (0.8초)



주문 데이터 로드 완료
  - 총 주문 수: 3,421,083
  - 고유 사용자 수: 206,209
  - 메모리 사용량: 254.4 MB
CPU times: total: 1.08 s
Wall time: 1.09 s


In [5]:
# 주문 데이터 샘플 확인
print("주문 데이터 컬럼 정보:")
print(orders_df.dtypes)
print("\n샘플 데이터:")
orders_df.head(10)

주문 데이터 컬럼 정보:
order_id                    int32
user_id                     int32
eval_set                   object
order_number                int16
order_dow                    int8
order_hour_of_day            int8
days_since_prior_order    float32
dtype: object

샘플 데이터:


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0
5,3367565,1,prior,6,2,7,19.0
6,550135,1,prior,7,1,9,20.0
7,3108588,1,prior,8,1,14,14.0
8,2295261,1,prior,9,1,16,0.0
9,2550362,1,prior,10,4,8,30.0


In [6]:
# eval_set 분포 확인
# - prior: 이전 주문 (학습용)
# - train: 학습 대상 주문
# - test: 테스트 주문 (상품 정보 없음)

print("eval_set 분포:")
print(orders_df['eval_set'].value_counts())

eval_set 분포:
eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64


## 4. 상품 데이터 로드

In [7]:
%%time

# 상품, 카테고리(aisle), 부서 데이터 로드
products_df = loader.load_products()
aisles_df = loader.load_aisles()
departments_df = loader.load_departments()

print(f"상품 데이터 로드 완료")
print(f"  - 총 상품 수: {len(products_df):,}")
print(f"  - 총 카테고리 수: {len(aisles_df):,}")
print(f"  - 총 부서 수: {len(departments_df):,}")

INFO:utils.personalization.data_processor:products.csv 로드 중...
INFO:utils.personalization.data_processor:  로드 완료: 49,688개 상품
INFO:utils.personalization.data_processor:aisles.csv 로드 완료: 134개 통로
INFO:utils.personalization.data_processor:departments.csv 로드 완료: 21개 부서


상품 데이터 로드 완료
  - 총 상품 수: 49,688
  - 총 카테고리 수: 134
  - 총 부서 수: 21
CPU times: total: 46.9 ms
Wall time: 37.1 ms


In [8]:
# 상품 데이터 샘플 확인
print("상품 데이터:")
products_df.head(10)

상품 데이터:


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13
5,6,Dry Nose Oil,11,11
6,7,Pure Coconut Water With Orange,98,7
7,8,Cut Russet Potatoes Steam N' Mash,116,1
8,9,Light Strawberry Blueberry Yogurt,120,16
9,10,Sparkling Orange Juice & Prickly Pear Beverage,115,7


In [9]:
# 카테고리(aisle) 데이터 확인
print(f"카테고리 수: {len(aisles_df)}")
print("\n샘플 카테고리:")
aisles_df.sample(20)

카테고리 수: 134

샘플 카테고리:


,aisle_id,aisle
51,52,frozen breakfast
79,80,deodorants
2,3,energy granola bars
37,38,frozen meals
94,95,canned meat seafood
29,30,latino foods
11,12,fresh pasta
20,21,packaged cheese
92,93,breakfast bakery
95,96,lunch meat


In [10]:
# 부서(department) 데이터 확인
print("부서 목록:")
departments_df

부서 목록:


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol
5,6,international
6,7,beverages
7,8,pets
8,9,dry goods pasta
9,10,bulk


## 5. 주문-상품 관계 로드 (대용량)

In [11]:
%%time

# order_products__prior.csv 로드 (대용량: 약 32M 행)
# 청크 단위로 로드하여 메모리 효율성 확보

order_products_prior_df = loader.load_order_products('prior')

print(f"\nprior 주문-상품 데이터 로드 완료")
print(f"  - 총 행 수: {len(order_products_prior_df):,}")
print(f"  - 고유 주문 수: {order_products_prior_df['order_id'].nunique():,}")
print(f"  - 고유 상품 수: {order_products_prior_df['product_id'].nunique():,}")
print(f"  - 메모리 사용량: {order_products_prior_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

INFO:utils.personalization.data_processor:order_products__prior.csv 로드 중... (d:\VSC_Project\SSAFY\SSAFY_Class_18_Team_4_Final_Capstone\SSAFY_Class_18_Team_4_Final_Capstone\data\instacart_raw\order_products__prior.csv)
INFO:utils.personalization.data_processor:  진행: 1,000,000개 레코드...
INFO:utils.personalization.data_processor:  진행: 2,000,000개 레코드...
INFO:utils.personalization.data_processor:  진행: 3,000,000개 레코드...
INFO:utils.personalization.data_processor:  진행: 4,000,000개 레코드...
INFO:utils.personalization.data_processor:  진행: 5,000,000개 레코드...
INFO:utils.personalization.data_processor:  진행: 6,000,000개 레코드...
INFO:utils.personalization.data_processor:  진행: 7,000,000개 레코드...
INFO:utils.personalization.data_processor:  진행: 8,000,000개 레코드...
INFO:utils.personalization.data_processor:  진행: 9,000,000개 레코드...
INFO:utils.personalization.data_processor:  진행: 10,000,000개 레코드...
INFO:utils.personalization.data_processor:  진행: 11,000,000개 레코드...
INFO:utils.personalization.data_processor:  진행: 12,000


prior 주문-상품 데이터 로드 완료
  - 총 행 수: 32,434,489
  - 고유 주문 수: 3,214,874
  - 고유 상품 수: 49,677
  - 메모리 사용량: 309.3 MB
CPU times: total: 6.03 s
Wall time: 6.08 s


In [12]:
%%time

# order_products__train.csv 로드
order_products_train_df = loader.load_order_products('train')

print(f"\ntrain 주문-상품 데이터 로드 완료")
print(f"  - 총 행 수: {len(order_products_train_df):,}")
print(f"  - 고유 주문 수: {order_products_train_df['order_id'].nunique():,}")
print(f"  - 고유 상품 수: {order_products_train_df['product_id'].nunique():,}")

INFO:utils.personalization.data_processor:order_products__train.csv 로드 중...
INFO:utils.personalization.data_processor:  로드 완료: 1,384,617개 레코드 (0.2초)



train 주문-상품 데이터 로드 완료
  - 총 행 수: 1,384,617
  - 고유 주문 수: 131,209
  - 고유 상품 수: 39,123
CPU times: total: 203 ms
Wall time: 178 ms


In [13]:
# 주문-상품 데이터 샘플 확인
print("주문-상품 데이터 컬럼:")
print(order_products_prior_df.dtypes)
print("\n샘플:")
order_products_prior_df.head(10)

주문-상품 데이터 컬럼:
order_id             int32
product_id           int32
add_to_cart_order     int8
reordered             int8
dtype: object

샘플:


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0
5,2,17794,6,1
6,2,40141,7,1
7,2,1819,8,1
8,2,43668,9,0
9,3,33754,1,1


## 6. 데이터 병합 및 전처리

In [14]:
%%time

# prior와 train 주문-상품 데이터 병합
order_products_df = pd.concat(
    [order_products_prior_df, order_products_train_df],
    ignore_index=True
)

# 메모리 정리
del order_products_prior_df, order_products_train_df

print(f"병합된 주문-상품 데이터")
print(f"  - 총 행 수: {len(order_products_df):,}")
print(f"  - 메모리 사용량: {order_products_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

병합된 주문-상품 데이터
  - 총 행 수: 33,819,106
  - 메모리 사용량: 322.5 MB
CPU times: total: 31.2 ms
Wall time: 49.1 ms


In [15]:
# 상품 데이터에 카테고리/부서 정보 병합
products_full_df = products_df.merge(
    aisles_df, on='aisle_id', how='left'
).merge(
    departments_df, on='department_id', how='left'
)

print(f"상품 데이터 (카테고리/부서 포함)")
print(f"  - 컬럼: {list(products_full_df.columns)}")
products_full_df.head(10)

상품 데이터 (카테고리/부서 포함)
  - 컬럼: ['product_id', 'product_name', 'aisle_id', 'department_id', 'aisle', 'department']


,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry
5,6,Dry Nose Oil,11,11,cold flu allergy,personal care
6,7,Pure Coconut Water With Orange,98,7,juice nectars,beverages
7,8,Cut Russet Potatoes Steam N' Mash,116,1,frozen produce,frozen
8,9,Light Strawberry Blueberry Yogurt,120,16,yogurt,dairy eggs
9,10,Sparkling Orange Juice & Prickly Pear Beverage,115,7,water seltzer sparkling water,beverages


## 7. 데이터 타입 최적화

In [16]:
def optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """
    DataFrame 데이터 타입 최적화
    
    - int64 → int32/int16/int8
    - float64 → float32
    - object → category
    """
    df_optimized = df.copy()
    
    for col in df_optimized.columns:
        col_type = df_optimized[col].dtype
        
        if col_type == 'int64':
            c_min = df_optimized[col].min()
            c_max = df_optimized[col].max()
            
            if c_min >= 0:
                if c_max < 256:
                    df_optimized[col] = df_optimized[col].astype('uint8')
                elif c_max < 65536:
                    df_optimized[col] = df_optimized[col].astype('uint16')
                elif c_max < 4294967296:
                    df_optimized[col] = df_optimized[col].astype('uint32')
            else:
                if c_min > -128 and c_max < 128:
                    df_optimized[col] = df_optimized[col].astype('int8')
                elif c_min > -32768 and c_max < 32768:
                    df_optimized[col] = df_optimized[col].astype('int16')
                else:
                    df_optimized[col] = df_optimized[col].astype('int32')
                    
        elif col_type == 'float64':
            df_optimized[col] = df_optimized[col].astype('float32')
            
        elif col_type == 'object':
            num_unique = df_optimized[col].nunique()
            num_total = len(df_optimized[col])
            if num_unique / num_total < 0.5:  # 카테고리화 기준: 고유값 비율 50% 미만
                df_optimized[col] = df_optimized[col].astype('category')
    
    return df_optimized

print("데이터 타입 최적화 함수 정의 완료")

데이터 타입 최적화 함수 정의 완료


In [17]:
%%time

# 메모리 사용량 비교
def get_memory_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2

print("최적화 전 메모리 사용량:")
print(f"  orders: {get_memory_mb(orders_df):.1f} MB")
print(f"  order_products: {get_memory_mb(order_products_df):.1f} MB")
print(f"  products: {get_memory_mb(products_full_df):.1f} MB")

# 최적화 적용
orders_optimized = optimize_dtypes(orders_df)
order_products_optimized = optimize_dtypes(order_products_df)
products_optimized = optimize_dtypes(products_full_df)
aisles_optimized = optimize_dtypes(aisles_df)
departments_optimized = optimize_dtypes(departments_df)

print("\n최적화 후 메모리 사용량:")
print(f"  orders: {get_memory_mb(orders_optimized):.1f} MB")
print(f"  order_products: {get_memory_mb(order_products_optimized):.1f} MB")
print(f"  products: {get_memory_mb(products_optimized):.1f} MB")

최적화 전 메모리 사용량:
  orders: 254.4 MB
  order_products: 322.5 MB
  products: 11.0 MB

최적화 후 메모리 사용량:
  orders: 55.5 MB
  order_products: 322.5 MB
  products: 4.7 MB
CPU times: total: 578 ms
Wall time: 573 ms


In [18]:
# 최적화된 데이터 타입 확인
print("최적화된 orders 데이터 타입:")
print(orders_optimized.dtypes)

print("\n최적화된 order_products 데이터 타입:")
print(order_products_optimized.dtypes)

최적화된 orders 데이터 타입:
order_id                     int32
user_id                      int32
eval_set                  category
order_number                 int16
order_dow                     int8
order_hour_of_day             int8
days_since_prior_order     float32
dtype: object

최적화된 order_products 데이터 타입:
order_id             int32
product_id           int32
add_to_cart_order     int8
reordered             int8
dtype: object


## 8. Parquet 형식으로 저장

In [19]:
%%time

# Parquet 형식으로 저장 (압축: snappy)
# Parquet 장점:
# - 열 기반 저장: 분석 쿼리에 효율적
# - 압축 효율: CSV 대비 50-80% 용량 절감
# - 스키마 보존: 데이터 타입 유지

orders_optimized.to_parquet(
    INSTACART_OUTPUT_PATH / 'orders.parquet',
    index=False,
    compression='snappy'
)
print(f"✓ orders.parquet 저장 완료")

order_products_optimized.to_parquet(
    INSTACART_OUTPUT_PATH / 'order_products.parquet',
    index=False,
    compression='snappy'
)
print(f"✓ order_products.parquet 저장 완료")

products_optimized.to_parquet(
    INSTACART_OUTPUT_PATH / 'products.parquet',
    index=False,
    compression='snappy'
)
print(f"✓ products.parquet 저장 완료")

aisles_optimized.to_parquet(
    INSTACART_OUTPUT_PATH / 'aisles.parquet',
    index=False,
    compression='snappy'
)
print(f"✓ aisles.parquet 저장 완료")

departments_optimized.to_parquet(
    INSTACART_OUTPUT_PATH / 'departments.parquet',
    index=False,
    compression='snappy'
)
print(f"✓ departments.parquet 저장 완료")

✓ orders.parquet 저장 완료
✓ order_products.parquet 저장 완료
✓ products.parquet 저장 완료
✓ aisles.parquet 저장 완료
✓ departments.parquet 저장 완료
CPU times: total: 1.98 s
Wall time: 1.97 s


In [20]:
# 저장된 파일 크기 확인
print("\n저장된 Parquet 파일:")
total_size = 0
for file in INSTACART_OUTPUT_PATH.glob('*.parquet'):
    size = file.stat().st_size / 1024**2
    total_size += size
    print(f"  {file.name}: {size:.1f} MB")

print(f"\n총 용량: {total_size:.1f} MB")


저장된 Parquet 파일:
  aisles.parquet: 0.0 MB
  departments.parquet: 0.0 MB
  mapped_products.parquet: 1.5 MB
  orders.parquet: 24.0 MB
  order_products.parquet: 116.8 MB
  products.parquet: 1.5 MB

총 용량: 143.8 MB


## 9. 데이터 검증

In [21]:
# 저장된 데이터 다시 로드하여 검증
orders_check = pd.read_parquet(INSTACART_OUTPUT_PATH / 'orders.parquet')
order_products_check = pd.read_parquet(INSTACART_OUTPUT_PATH / 'order_products.parquet')
products_check = pd.read_parquet(INSTACART_OUTPUT_PATH / 'products.parquet')

print("데이터 검증:")
print(f"  orders 행 수: {len(orders_check):,} (원본: {len(orders_optimized):,})")
print(f"  order_products 행 수: {len(order_products_check):,} (원본: {len(order_products_optimized):,})")
print(f"  products 행 수: {len(products_check):,} (원본: {len(products_optimized):,})")

# 무결성 검증
assert len(orders_check) == len(orders_optimized), "orders 행 수 불일치"
assert len(order_products_check) == len(order_products_optimized), "order_products 행 수 불일치"
assert len(products_check) == len(products_optimized), "products 행 수 불일치"

print("\n✓ 모든 검증 통과")

데이터 검증:
  orders 행 수: 3,421,083 (원본: 3,421,083)
  order_products 행 수: 33,819,106 (원본: 33,819,106)
  products 행 수: 49,688 (원본: 49,688)

✓ 모든 검증 통과


## 10. 요약 통계

In [22]:
# 최종 요약 통계
summary = {
    '총 주문 수': f"{len(orders_check):,}",
    '총 사용자 수': f"{orders_check['user_id'].nunique():,}",
    '총 상품 수': f"{len(products_check):,}",
    '총 카테고리 수': f"{products_check['aisle_id'].nunique():,}",
    '총 부서 수': f"{products_check['department_id'].nunique():,}",
    '총 주문-상품 관계': f"{len(order_products_check):,}",
    '주문당 평균 상품 수': f"{len(order_products_check) / len(orders_check):.1f}",
    '사용자당 평균 주문 수': f"{len(orders_check) / orders_check['user_id'].nunique():.1f}",
}

print("="*50)
print("Instacart 데이터셋 요약")
print("="*50)
for key, value in summary.items():
    print(f"{key}: {value}")
print("="*50)

Instacart 데이터셋 요약
총 주문 수: 3,421,083
총 사용자 수: 206,209
총 상품 수: 49,688
총 카테고리 수: 134
총 부서 수: 21
총 주문-상품 관계: 33,819,106
주문당 평균 상품 수: 9.9
사용자당 평균 주문 수: 16.6


In [23]:
# 메타데이터 저장
import json

metadata = {
    'source': 'Kaggle Instacart Market Basket Analysis',
    'processed_at': datetime.now().isoformat(),
    'statistics': {
        'n_orders': int(len(orders_check)),
        'n_users': int(orders_check['user_id'].nunique()),
        'n_products': int(len(products_check)),
        'n_aisles': int(products_check['aisle_id'].nunique()),
        'n_departments': int(products_check['department_id'].nunique()),
        'n_order_products': int(len(order_products_check)),
    },
    'files': [
        'orders.parquet',
        'order_products.parquet',
        'products.parquet',
        'aisles.parquet',
        'departments.parquet',
    ],
}

with open(INSTACART_OUTPUT_PATH / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("메타데이터 저장 완료: metadata.json")

메타데이터 저장 완료: metadata.json


## 다음 단계

- **21_instacart_eda_analysis.ipynb**: 탐색적 데이터 분석
  - 사용자 행동 패턴 분석
  - 상품 인기도 분포
  - 재구매율 분석
  - 시간대별 구매 패턴